# Irrigation Training — **v2.22** (Markov-r5: prev_u + 9-feature network)

v2.22 = **v2.21c** (additive terminal-yield — the seed-validated best controller: MPC yield parity, beats MPC on waterlog/water, mod/70% fixed) **+ prev_u**: `u_{t-1}` is appended as a 9th per-agent observation feature so the delta-u smoothing reward **r5 becomes Markov**. This is the **one** new variable vs v2.21c. It targets the only remaining gap — **pulsing** (mean|Δu| ~2.35 vs MPC's 0.97).

Uses the 9-feature actor+critic (`networks_td3_prevu.py`), byte-identical to v2.21c's architecture (per-agent shared MLP, 2×−1 re-center, VDN sum, twin-Q, LayerNorm critic) **except the per-agent input width (8→9)**. `gym_env.py` (v2.21c) and `gym_env_prev_u.py` are unchanged and already in the repo. SAC / v2.20 / v2.21c are unaffected.

**`reward_du_alpha` stays at 0.005** (its value when r5 was non-Markov). If pulsing doesn't drop, raise it (0.02–0.05) in `configs_v222.py` — a one-line change.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE_ROOT='/content/drive/MyDrive/thesis_v220_td3_runs'; os.makedirs(DRIVE_ROOT,exist_ok=True)
print('Drive mounted:',DRIVE_ROOT)


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0). Same stack as the SAC/TD3 runs.
import subprocess, sys, os
WORK='/content'; repo=os.path.join(WORK,'thesis')
if os.path.exists(repo): subprocess.run(['rm','-rf',repo],check=True)
subprocess.run(['git','clone','https://github.com/taratorbati/thesis.git',repo],check=True)
os.chdir(repo); sys.path.insert(0,repo)
subprocess.run(['pip','install','--quiet','stable-baselines3==2.6.0','gymnasium','wandb','pytest'],check=True)
import torch; print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY'); print('OK WANDB key.')
except Exception as e:
    try:
        import getpass; k=getpass.getpass('WANDB key (Enter to skip): ').strip()
        if k: os.environ['WANDB_API_KEY']=k; print('OK set.')
        else: print('Skipping WandB.')
    except Exception: print('Skipping WandB.')
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


## [OPTIONAL] Write v2.22 files

Writes the 3 NEW files (`networks_td3_prevu.py`, `configs_v222.py`, `train_v222_td3.py`). Skip if already pushed to the repo.


In [ ]:
# [OPTIONAL] Write the 3 NEW v2.22 files into the cloned repo. Skip if already pushed.
# gym_env.py (v2.21c) and gym_env_prev_u.py are UNCHANGED and already in the repo.
from pathlib import Path
REPO = '/content/thesis'
_files = {
    'src/rl/networks_td3_prevu.py':
        '23207372632f726c2f6e6574776f726b735f7464335f70726576752e7079202076322e32322020284d61726b6f762d7235202f20707265765f753a20'
        '392d6665617475726520544433206e6574776f726b290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2320392d66656174757265205444332056'
        '444e206163746f72202b2063726974696320666f722049727269676174696f6e456e7650726576552c20776869636820617070656e6473207468650a'
        '232070726576696f7573206170706c69656420636f6e74726f6c20755f7b742d317d206173206120397468207065722d6167656e74206f6273657276'
        '6174696f6e206665617475726520736f207468650a232064656c74612d7520736d6f6f7468696e6720726577617264207235206265636f6d6573204d'
        '61726b6f762028746865206167656e742063616e2066696e616c6c7920534545207468650a23207175616e746974792069747320736d6f6f74686e65'
        '73732070656e616c747920646570656e6473206f6e292e0a230a232054686973206d6f64756c6520737562636c6173736573207468652076322e3139'
        '2d544433206163746f7220285f5444335368617265644163746f722920616e64207468652076322e31310a23204c617965724e6f726d2056444e2063'
        '726974696320285f56323131466163746f72697a6564514e6574202f205f56323131466163746f72697a6564436f6e74696e756f7573437269746963'
        '290a2320616e64206f7665727269646573204f4e4c5920746865207065722d6167656e742064696d656e73696f6e733a0a2320202020207065722d61'
        '67656e7420666561747572657320202020202038202d3e2039202020284e5f4147454e545f4645415455524553290a2320202020206163746f722070'
        '65722d6167656e7420696e70757420203635202d3e2036362020285045525f4147454e545f494e5055545f44494d2020202020203d2039202b203537'
        '20676c6f62616c73290a232020202020637269746963207065722d6167656e7420696e707574203636202d3e2036372020285045525f4147454e545f'
        '4352495449435f494e5055545f44494d203d2039202b203537202b203120616374696f6e290a23202020202066756c6c206f62732064696d20202020'
        '202020202031303937202d3e20313232370a23205468652061726368697465637475726520287065722d6167656e7420736861726564204d4c502c20'
        '32782d312072652d63656e746572696e672c206167656e742d6d616a6f7220726573686170652c0a232056444e2073756d2c207477696e2d512c204c'
        '617965724e6f726d2d61667465722d4c696e6561722920697320627974652d6964656e746963616c20746f2076322e323163202d2d206f6e6c792074'
        '68650a2320696e707574207769647468206368616e6765732e2054686520392d6665617475726520636f6e7374616e747320616c7265616479206578'
        '69737420696e206e6574776f726b732e70793b2077650a2320646f204e4f5420746f7563682074686520736861726564205632375f2a2028382d6665'
        '61747572652920636f6e7374616e7473206f7220636c61737365732c20736f20746865205341430a232066616d696c7920616e64207468652076322e'
        '32302f76322e32316320382d666561747572652054443320706174682061726520636f6d706c6574656c7920756e61666665637465642e0a23202d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a66726f6d20747970'
        '696e6720696d706f7274204f7074696f6e616c0a0a66726f6d20737461626c655f626173656c696e6573332e636f6d6d6f6e2e706f6c696369657320'
        '696d706f727420426173654665617475726573457874726163746f720a66726f6d20737461626c655f626173656c696e6573332e636f6d6d6f6e2e70'
        '726570726f63657373696e6720696d706f7274206765745f616374696f6e5f64696d0a0a66726f6d207372632e726c2e6e6574776f726b7320696d70'
        '6f727420280a202020204e5f4147454e545f46454154555245532c202020202020202020202020202320390a202020205045525f4147454e545f494e'
        '5055545f44494d2c20202020202020202020232036362020283d2039202b203537290a202020205045525f4147454e545f4352495449435f494e5055'
        '545f44494d2c202020232036372020283d2039202b203537202b2031290a202020205f56323131466163746f72697a6564514e65742c0a202020205f'
        '56323131466163746f72697a6564436f6e74696e756f75734372697469632c0a290a66726f6d207372632e726c2e6e6574776f726b735f7464332069'
        '6d706f7274205f5444335368617265644163746f722c2054443356444e506f6c6963792c206d616b655f7464335f706f6c6963795f6b77617267730a'
        '0a5f5f616c6c5f5f203d205b2254443356444e506f6c6963795072657655222c20226d616b655f7464335f706f6c6963795f6b7761726773225d0a0a'
        '0a636c617373205f5444335368617265644163746f725072657655285f5444335368617265644163746f72293a0a2020202022222276322e32316320'
        '64657465726d696e69737469632056444e206163746f7220776974682039207065722d6167656e7420666561747572657320286164647320755f7b74'
        '2d317d292e2222220a202020205f4e5f4147454e545f4645415455524553202020203d204e5f4147454e545f46454154555245532020202020202020'
        '2320390a202020205f5045525f4147454e545f494e5055545f44494d203d205045525f4147454e545f494e5055545f44494d2020202020232036360a'
        '0a0a636c617373205f56323131466163746f72697a6564514e65745072657655285f56323131466163746f72697a6564514e6574293a0a2020202022'
        '222276322e3131204c617965724e6f726d2056444e20512d6e657420776974682039207065722d6167656e7420666561747572657320286372697469'
        '6320696e707574203637292e2222220a202020205f4e5f4147454e545f464541545552455320202020202020202020203d204e5f4147454e545f4645'
        '4154555245532020202020202020202020202320390a202020205f5045525f4147454e545f4352495449435f494e5055545f44494d203d205045525f'
        '4147454e545f4352495449435f494e5055545f44494d2020232036370a0a0a636c617373205f56323131466163746f72697a6564436f6e74696e756f'
        '75734372697469635072657655285f56323131466163746f72697a6564436f6e74696e756f7573437269746963293a0a202020202222224c61796572'
        '4e6f726d207477696e2d512056444e2063726974696320776974682039207065722d6167656e742066656174757265732e2222220a202020205f4e5f'
        '4147454e545f4645415455524553203d204e5f4147454e545f46454154555245532020202020202020202020202320390a202020205f514e45545f43'
        '4c532020202020202020203d205f56323131466163746f72697a6564514e657450726576550a0a0a636c6173732054443356444e506f6c6963795072'
        '6576552854443356444e506f6c696379293a0a202020202222225444332056444e20706f6c69637920666f722074686520392d666561747572652070'
        '7265765f75206f62736572766174696f6e2028313232372d64696d292e0a0a202020204964656e746963616c20746f2054443356444e506f6c696379'
        '20657863657074206974206275696c64732074686520392d66656174757265206163746f72202b206372697469632e0a20202020706f6c6963795f6b'
        '776172677320617265207468652073616d652061732074686520382d6665617475726520706f6c6963792028746865206665617475726520636f756e'
        '74206c697665730a20202020696e20746865206163746f722f63726974696320636c61737365732c206e6f7420746865206b7761726773292c20736f'
        '207265757365206d616b655f7464335f706f6c6963795f6b77617267732e0a202020202222220a0a20202020646566206d616b655f6163746f722873'
        '656c662c2066656174757265735f657874726163746f723a204f7074696f6e616c5b426173654665617475726573457874726163746f725d203d204e'
        '6f6e6529202d3e205f5444335368617265644163746f7250726576553a0a20202020202020206163746f725f6b7761726773203d2073656c662e5f75'
        '70646174655f66656174757265735f657874726163746f722873656c662e6163746f725f6b77617267732c2066656174757265735f65787472616374'
        '6f72290a20202020202020206163746f725f6b77617267735b224e225d203d206765745f616374696f6e5f64696d2873656c662e616374696f6e5f73'
        '70616365290a202020202020202072657475726e205f5444335368617265644163746f725072657655282a2a6163746f725f6b7761726773292e746f'
        '2873656c662e646576696365290a0a20202020646566206d616b655f6372697469632873656c662c2066656174757265735f657874726163746f723a'
        '204f7074696f6e616c5b426173654665617475726573457874726163746f725d203d204e6f6e6529202d3e205f56323131466163746f72697a656443'
        '6f6e74696e756f757343726974696350726576553a0a20202020202020206372697469635f6b7761726773203d2073656c662e5f7570646174655f66'
        '656174757265735f657874726163746f722873656c662e6372697469635f6b77617267732c2066656174757265735f657874726163746f72290a2020'
        '2020202020206372697469635f6b77617267735b224e225d203d206765745f616374696f6e5f64696d2873656c662e616374696f6e5f737061636529'
        '0a202020202020202072657475726e205f56323131466163746f72697a6564436f6e74696e756f75734372697469635072657655282a2a6372697469'
        '635f6b7761726773292e746f2873656c662e646576696365290a'
        ,
    'src/rl/configs_v222.py':
        '23207372632f726c2f636f6e666967735f763232322e7079202076322e32322020284d61726b6f762d72353a20707265765f75202b20392d66656174'
        '757265206e6574776f726b290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a232076322e3232203d2076322e3231632028616464697469766520'
        '7465726d696e616c2d7969656c64202d2d2074686520736565642d76616c696461746564206265737420636f6e74726f6c6c65723a0a23204d504320'
        '7969656c64207061726974792c206265617473204d5043206f6e2077617465726c6f672f77617465722c206d6f642f37302520666978656429202b20'
        '707265765f753a20755f7b742d317d0a2320697320616464656420746f20746865207065722d6167656e74206f62736572766174696f6e20736f2074'
        '68652064656c74612d7520736d6f6f7468696e6720726577617264207235206265636f6d65730a23204d41524b4f562e205468697320697320746865'
        '204f4e45206e6577207661726961626c652076732076322e3231632e204974207461726765747320746865206f6e6c792072656d61696e696e670a23'
        '20676170202d2d2050554c53494e4720286d65616e7c64757c207e322e3335207673204d5043277320302e3937293b20723520636f756c64206e6f74'
        '20776f726b206265666f726520626563617573650a2320612064657465726d696e6973746963206163746f7220636f756c64206e6f74206f62736572'
        '766520755f7b742d317d2e0a230a2320557365732074686520392d66656174757265206163746f722b63726974696320286e6574776f726b735f7464'
        '335f7072657675292c20627974652d6964656e746963616c20746f2076322e32316327730a2320617263686974656374757265206578636570742074'
        '6865207065722d6167656e7420696e7075742077696474682028382d3e39206665617475726573292e20416c6c2076322e323163207265776172640a'
        '232073657474696e677320617265206b657074202862696f6d6173735f73686170696e673d46616c73652c207265776172645f7465726d696e616c5f'
        '7969656c643d312e30292e0a230a23207265776172645f64755f616c70686120737461797320617420302e30303520287468652076616c7565207365'
        '74207768656e20723520776173204e4f4e2d4d61726b6f762c20692e652e20696e657274292e0a23204e6f77207468617420746865206167656e7420'
        '63616e20616374206f6e2069742c20302e303035206d617920616c726561647920626974653b2069662070756c73696e6720646f6573204e4f540a23'
        '2064726f70206d65616e696e6766756c6c792c20746865206e657874206c657665722069732072616973696e67207265776172645f64755f616c7068'
        '61202874727920302e30322d302e303529202d2d0a232061206f6e652d6c696e65206368616e676520696e20746869732066696c652c206e6f206f74'
        '6865722066696c6573206e65656465642e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d'
        '2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a66726f6d205f5f6675747572655f5f20696d706f7274'
        '20616e6e6f746174696f6e730a0a47414d4d415f42415345203d20302e39390a0a52554e5f41203d2064696374280a202020206c6162656c3d226d61'
        '726b6f767235222c0a2020202023202d2d2d20696e686572697465642066726f6d2076322e323163202876616c696461746564206265737429202d2d'
        '2d0a202020206e5f73746570733d352c0a2020202067616d6d615f626173653d47414d4d415f424153452c0a202020206c6561726e696e675f737461'
        '7274733d35305f3030302c0a202020207265776172645f64755f616c7068613d302e3030352c2020202020202020202023207235207765696768743b'
        '206e6f7720414354494f4e41424c452028707265765f7520696e206f6273290a20202020706f6c6963795f64656c61793d322c0a2020202074617267'
        '65745f706f6c6963795f6e6f6973653d302e322c0a202020207461726765745f6e6f6973655f636c69703d302e352c0a202020206163746f725f6c72'
        '5f6d756c743d312e302c0a202020206163746f725f7761726d75705f757064617465733d302c0a2020202062696f6d6173735f73686170696e673d46'
        '616c73652c2020202020202020202023206b656570207468652064656e736520696e6372656d656e742072310a202020207265776172645f7465726d'
        '696e616c5f7969656c643d312e302c20202020202023206b65657020746865206164646974697665207465726d696e616c2d7969656c64207465726d'
        '0a2020202023202d2d2d20746865204f4e452076322e3232206368616e6765202d2d2d0a202020206578706f73655f707265765f753d547275652c20'
        '20202020202020202020202023202d3e2049727269676174696f6e456e765072657655202b20392d66656174757265206e6574776f726b0a290a0a43'
        '4f4e46494753203d207b2241223a2052554e5f417d0a'
        ,
    'src/rl/train_v222_td3.py':
        '...'
        '20290a'
        ,
}
for relpath, hexstr in _files.items():
    p = Path(REPO) / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(bytes.fromhex(hexstr))
    print(f'  written: {relpath} ({p.stat().st_size:,} bytes)')
print('v2.22 files ready (networks_td3_prevu + configs_v222 + train_v222_td3).')


In [ ]:
# Smoke tests + a v2.21 pilot that runs real gradient steps.
# Catches import errors and training-loop bugs before committing ~1 hr of GPU time.
import subprocess, sys
REPO = '/content/thesis'
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short'],cwd=REPO).returncode==0,'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short'],cwd=REPO).returncode==0,'CRITIC TESTS FAILED'
print('\nv2.21 pilot (runs real gradient steps with NStepReplayBufferExact)...')
from src.rl import configs_v222
import copy
_pilot = copy.deepcopy(configs_v222.CONFIGS['A'])
_pilot['learning_starts'] = 200  # small enough for 1200-step pilot
configs_v222.CONFIGS['PILOT'] = _pilot
from src.rl.train_v222_td3 import train_td3_v222
m = train_td3_v222('PILOT', seed=999, output_dir='/content/pilot', total_timesteps=1200)
import glob, zipfile, io, torch
ck = glob.glob('/content/pilot/td3_v222_markovr5_seed999/*final*.zip')
if ck:
    with zipfile.ZipFile(ck[0]) as z:
        sd = torch.load(io.BytesIO(z.open('policy.pth').read()), map_location='cpu', weights_only=False)
    assert 'actor.log_std.weight' not in sd, 'BUG: log_std found (SAC checkpoint loaded?)'
    assert 'actor.mu_head.weight' in sd, 'BUG: mu_head missing'
    assert abs(float(sd['actor.obs_norm_marker'].item()) - 2.19) < 0.01, 'BUG: marker != 2.19'
    print('  Checkpoint: deterministic actor, mu_head present, marker=2.19. OK')
assert type(m.replay_buffer).__name__ == 'NStepReplayBufferExact', 'BUG: wrong buffer type'
assert m.replay_buffer.n_steps == 5, 'BUG: wrong n_steps'
print(f'  Buffer: {type(m.replay_buffer).__name__}  n_steps={m.replay_buffer.n_steps}  gamma={m.replay_buffer._n_gamma}. OK')
print(f'  model.gamma = {m.gamma:.6f}  (expect {0.99**5:.6f}). {"OK" if abs(m.gamma - 0.99**5)<1e-8 else "BUG"}')
print('\nOK pre-flight passed. Proceed to training.')


In [ ]:
# Full 250k TD3 v2.22 (additive terminal-yield on the v2.20 Run A increment reward).
# Keeps the full dense r1; adds alpha_T*x4_final/X4_REF once at episode end. ~1-1.5 hr T4.
SEED = 0    # CHANGE per session
REPO = '/content/thesis'
from src.rl.train_v222_td3 import train_td3_v222
model = train_td3_v222(
    config_name='A',
    seed=SEED,
    output_dir=f'{REPO}/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    # ── params from CONFIGS['A'] in configs_v222.py ──
    # INHERITED FROM v2.20 RUN A (unchanged): n_steps=5, gamma_base=0.99,
    #   model.gamma=0.99**5, learning_starts=50_000, reward_du_alpha=0.005, warmup off.
    # SCALES reverted in gym_env.py: X4_REF=600, ALPHA2=0.016 (v2.20 Run A values).
    # THE ONLY v2.22 CHANGE:
    #   biomass_shaping=False      -> KEEP the dense increment r1 (NOT gamma-shaped)
    #   reward_terminal_yield=1.0  -> ADD alpha_T*x4_final/X4_REF once at episode end
    #                                 (season-sum calibrated; tune [0.5,1.5]).
)
print('Training complete.')

In [ ]:
import shutil, os, datetime
RUN_LABEL = 'markovr5'   # 'nstep5_damped' for Run B
src=f'/content/thesis/results/rl/td3_v222_{RUN_LABEL}_seed{SEED}'
dst=os.path.join(DRIVE_ROOT,f'td3_v222_{RUN_LABEL}_seed{SEED}_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree(src,dst,ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to Drive:',dst)


In [ ]:
# Post-training eval: best_model + final model, perfect + noisy forecasts.
# Writes parquets to results/runs/<tag>/ for the comparison diagnostic.
import subprocess, sys, os
REPO = '/content/thesis'
RUN_LABEL = 'markovr5'   # match what you ran
run_dir = f'{REPO}/results/rl/td3_v222_{RUN_LABEL}_seed{SEED}'
model_path = f'{run_dir}/best_model/best_model.zip'
final_path = f'{run_dir}/td3_v222_{RUN_LABEL}_seed{SEED}_final.zip'
BEST_TAG=f'td3_v222_{RUN_LABEL}_best_seed{SEED}'; FINAL_TAG=f'td3_v222_{RUN_LABEL}_final_seed{SEED}'
print('Evaluating BEST (perfect)...')
r=subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','perfect',
    '--force','--out-tag',BEST_TAG],capture_output=True,text=True,cwd=REPO)
print(r.stdout[-1500:])
if r.returncode!=0: print('STDERR:',r.stderr[-2500:])
assert r.returncode==0,'PERFECT EVAL FAILED'
print('\nEvaluating BEST (noisy, forecast-sensitivity check)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','noisy',
    '--noise-seed','42','--force','--out-tag',BEST_TAG],cwd=REPO)
if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k, perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
        '--model',final_path,'--scenario','all','--budget','all','--forecast','perfect',
        '--force','--out-tag',FINAL_TAG],cwd=REPO)
print(f'\nBest-model eval -> results/runs/{BEST_TAG}/')


In [ ]:
# PRIMARY v2.22 DIAGNOSTIC: did Markov-r5 (prev_u + 9-feature net) cut PULSING
# toward MPC -- WITHOUT losing the v2.21c gains (yield parity, waterlog, mod/70%)?
import pandas as pd, numpy as np, json, glob, os
REPO='/content/thesis'; RUN_LABEL='markovr5'
OUT=f'{REPO}/results/runs/td3_v222_{RUN_LABEL}_best_seed{SEED}'
MPC=f'{REPO}/results/runs'
assert os.path.isdir(OUT), f'{OUT} missing -- run the eval cell first.'
SC=['dry','moderate','wet']; BD=['100','85','70']

def jm(d,scen,b,kind):
    g=(glob.glob(os.path.join(d,f'sac_perfect_det_{scen}_rice_{b}pct_seed*.json')) if kind=='rl'
       else glob.glob(os.path.join(d,f'mpc_perfect_{scen}_rice_{b}pct_Hp8.json')))
    return json.load(open(g[0]))['final_metrics'] if g else None
def pq(d,scen,b,kind):
    g=(glob.glob(os.path.join(d,f'sac_perfect_det_{scen}_rice_{b}pct_seed*.parquet')) if kind=='rl'
       else glob.glob(os.path.join(d,f'mpc_perfect_{scen}_rice_{b}pct_Hp8.parquet')))
    return pd.read_parquet(g[0]) if g else None
def pulse(df):
    return df.sort_values('day').groupby('agent')['u'].apply(lambda x: x.diff().abs().mean()).mean()

# ---- 1) PULSING (the v2.22 target) ----
print('='*84)
print('PULSING  mean|du| (mm/day, lower=smoother)')
print(f'{"cell":<13}{"v2.22":>9}{"MPC":>9}')
pv,pm=[],[]
for scen in SC:
    for b in BD:
        a,m=pq(OUT,scen,b,'rl'),pq(MPC,scen,b,'mpc')
        if a is not None and m is not None:
            va,vm=pulse(a),pulse(m); pv.append(va); pm.append(vm)
            print(f'{scen+"/"+b+"%":<13}{va:>9.2f}{vm:>9.2f}')
print('-'*84)
print(f'9-CELL MEAN  pulsing: v2.22={np.mean(pv):.2f}  MPC={np.mean(pm):.2f}   (v2.21c reference: 2.35)')
print(f'  -> drop from 2.35 toward {np.mean(pm):.2f} is the v2.22 success signal.')

# ---- 2) confirm v2.21c gains held: yield / drought / waterlog vs MPC ----
print('='*84)
print(f'{"cell":<13}{"v2.22 Y":>8}{"MPC Y":>8}{"d":>6} | {"v2.22 dr":>9}{"MPC dr":>8} | {"wlog":>7}{"MPC":>6}')
yA,yM,dA,dM,wA,wM=[],[],[],[],[],[]
for scen in SC:
    for b in BD:
        a,m=jm(OUT,scen,b,'rl'),jm(MPC,scen,b,'mpc')
        if a and m:
            yA.append(a['yield_kg_ha']); yM.append(m['yield_kg_ha'])
            dA.append(a['drought_days_per_agent']); dM.append(m['drought_days_per_agent'])
            wA.append(a['waterlog_days_per_agent']); wM.append(m['waterlog_days_per_agent'])
            print(f'{scen+"/"+b+"%":<13}{a["yield_kg_ha"]:>8.0f}{m["yield_kg_ha"]:>8.0f}'
                  f'{a["yield_kg_ha"]-m["yield_kg_ha"]:>+6.0f} | '
                  f'{a["drought_days_per_agent"]:>9.1f}{m["drought_days_per_agent"]:>8.1f} | '
                  f'{a["waterlog_days_per_agent"]:>7.1f}{m["waterlog_days_per_agent"]:>6.1f}')
print('-'*84)
ay,my=np.mean(yA),np.mean(yM)
print(f'9-CELL MEAN  yield: v2.22={ay:.0f} ({100*ay/my:.1f}% MPC)   drought-d: {np.mean(dA):.1f} (MPC {np.mean(dM):.1f})   waterlog-d: {np.mean(wA):.1f} (MPC {np.mean(wM):.1f})')
print(f'  v2.21c REFERENCE:  yield 3800 (99.74%)   waterlog 5.9-7.5 (<=MPC 7.7)   mod/70% gap ~ -40')
m70=jm(OUT,'moderate','70','rl'); mm=jm(MPC,'moderate','70','mpc')
print(f'  mod/70%:  yield v2.22={m70["yield_kg_ha"]:.0f} vs MPC {mm["yield_kg_ha"]:.0f} (gap {m70["yield_kg_ha"]-mm["yield_kg_ha"]:+.0f}; v2.21c was ~ -40)')
print('='*84)
print('GATE: pulsing DROPS from 2.35 toward MPC  AND  yield >= 99.7% (held v2.21c)')
print('      AND  waterlog <= MPC (7.7)  AND  mod/70% gap stays small.')
print('If pulsing barely moves: raise reward_du_alpha (0.005 -> 0.02-0.05) in configs_v222.py and rerun.')


In [ ]:
# STABILITY DIAGNOSTIC. q_pred calibration + collapse guard + coverage.
# Stage-1 success = q_pred BOUNDED (not monotone), guard never trips.
import os, glob, numpy as np, pandas as pd
REPO = '/content/thesis'
RUN_LABEL = 'markovr5'
run_dir=f'{REPO}/results/rl/td3_v222_{RUN_LABEL}_seed{SEED}'
br=os.path.join(run_dir,'bias_ratio_log.csv')
if os.path.exists(br):
    b=pd.read_csv(br); print('--- bias_ratio_log (q_pred trajectory) ---')
    print(b.to_string(index=False))
    q_min=float(b['q_pred_mean'].min()); q_final=float(b['q_pred_mean'].iloc[-1])
    q_mono = all(b['q_pred_mean'].diff().dropna() < 0)   # True if always decreasing
    print(f'\n  q_pred min={q_min:+.1f}  final={q_final:+.1f}  monotone_dive={q_mono}')
    if q_mono:
        verdict = 'FAIL -- monotone dive (same as v2.20 r5). Try Run B.'
    elif q_min < -30 and q_final >= -5:
        verdict = 'PARTIAL -- deep dip but recovered. Watch over seeds; consider Run B.'
    elif q_final >= -5:
        verdict = 'PASS -- q_pred bounded and recovered.'
    else:
        verdict = f'UNCERTAIN -- q_pred final={q_final:+.1f}. Check the curve.'
    print(f'  VERDICT: {verdict}')
else:
    print('No bias_ratio_log.csv at', br)

cg=os.path.join(run_dir,'collapse_guard_log.csv')
if os.path.exists(cg):
    g=pd.read_csv(cg)
    tripped=int(g['collapsed'].max()) if 'collapsed' in g.columns and len(g) else 0
    last_frac=g['frac_low_rolling'].iloc[-1] if len(g) else float('nan')
    print(f'\n--- collapse_guard ---  rows={len(g)}  final_frac_low={last_frac:.0%}')
    print(f'  guard tripped: {"YES -- collapsed" if tripped else "NO -- healthy"}')

cov=os.path.join(run_dir,'low_action_coverage_log.csv')
if os.path.exists(cov):
    c=pd.read_csv(cov)
    print(f'--- low_action_coverage ---  mean frac_low (last 50k)={c["frac_low_action"].tail(50).mean():.0%}')

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    runs=glob.glob(os.path.join(run_dir,'tensorboard','*'))
    if runs:
        ea=EventAccumulator(runs[0]); ea.Reload()
        if 'train/critic_loss' in ea.Tags()['scalars']:
            cl=ea.Scalars('train/critic_loss'); mxl=max(e.value for e in cl)
            print(f'\n  max critic_loss = {mxl:.2f}  (STABLE if < 100; v2.7 cascade hit 6.9e12)')
except Exception as e:
    print('tensorboard read skipped:', e)


## [NEXT VERSION] Spatial GNN actor

With pulsing addressed by v2.22's Markov-r5, the remaining architectural lever is a **GNN spatial actor** that lets cells share information along the D8 routing graph (rather than the current parameter-shared per-cell MLP). That is the largest change and the natural next step after confirming v2.22 reduces mean|Δu| toward MPC while holding the v2.21c yield/waterlog/mod-70% results.


In [ ]:
# [OPTIONAL] Resume v2.21 from a checkpoint (after a Colab disconnect).
# from src.rl.train_v222_td3 import WarmupAsymmetricLRTD3
# SEED=0; RUN_LABEL='markovr5'; STEP=150_000
# ckpt=f'/content/thesis/results/rl/td3_v222_{RUN_LABEL}_seed{SEED}/checkpoints/td3_v222_{RUN_LABEL}_seed{SEED}_{STEP}_steps.zip'